# 01. Preprocessament de Dades i Segmentació Temporal

En aquest quadern realitze l'extracció, neteja i preparació de les dades en brut provinents dels sensors i bombes d'insulina (fitxers XML). L'objectiu és generar sèries temporals regulars de 5 minuts, imputar xicotetes pèrdues de senyal i extraure exclusivament els segments continus que superen el llindar de 12 hores establit en l'anàlisi exploratoria prèvia.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import xml.etree.ElementTree as ET
import warnings

warnings.filterwarnings("ignore")
print("S'han carregat correctament totes les llibreries del sistema.")

# PARÀMETRES GLOBALS

# Llindar mínim de mostres per considerar un segment vàlid (144 mostres = 12 hores)
LLINDAR_TRAIN = 144   
LLINDAR_TEST  = 144   

RUTA_ARXIUS_TRAIN = "archive/*-ws-training.xml"
RUTA_ARXIUS_TEST  = "archive/*-ws-testing.xml"
RUTA_EIXIDA       = "dades_preprocessades"

os.makedirs(RUTA_EIXIDA, exist_ok=True)

S'han carregat correctament totes les llibreries del sistema.


### 1. Extracció XML i Remostratge Temporal

Definisc la funció encarregada de parsejar els arxius XML individuals. Extraig els valors de glucosa capil·lar, les ingestes de carbohidrats (`meal`) i les injeccions d'insulina ràpida (`bolus`). Posteriorment, remostrege la sèrie combinada a una freqüència de 5 minuts i aplique una imputació lineal per a rescatar buits de connexió curts (màxim 3 mostres consecutives o 15 minuts).

In [ ]:
def processar_arxiu_xml(ruta_arxiu):
    try:
        tree = ET.parse(ruta_arxiu)
        root = tree.getroot()

        # Glucosa
        glucose_section = root.find('glucose_level')
        data_glucose = [
            {"timestamp": ev.get("ts"), "glucose": ev.get("value")}
            for ev in glucose_section.findall('event')
        ] if glucose_section is not None else []
        df_g = pd.DataFrame(data_glucose)
        if df_g.empty:
            return None
        df_g['glucose'] = pd.to_numeric(df_g['glucose'], errors='coerce')

        # Carbohidrats (meal)
        meal_section = root.find('meal')
        data_meal = [
            {"timestamp": ev.get("ts"), "carbs": ev.get("carbs")}
            for ev in meal_section.findall('event')
        ] if meal_section is not None else []
        df_m = pd.DataFrame(data_meal)
        if not df_m.empty:
            df_m['carbs'] = pd.to_numeric(df_m['carbs'], errors='coerce')

        # Insulina ràpida (bolus)
        bolus_section = root.find('bolus')
        data_bolus = [
            {"timestamp": ev.get("ts_begin") or ev.get("ts"), "bolus": ev.get("dose")}
            for ev in bolus_section.findall('event')
        ] if bolus_section is not None else []
        df_b = pd.DataFrame(data_bolus)
        if not df_b.empty:
            df_b['bolus'] = pd.to_numeric(df_b['bolus'], errors='coerce')

        # Conversió de timestamps i indexació
        for df in [df_g, df_m, df_b]:
            if not df.empty:
                df['timestamp'] = pd.to_datetime(df['timestamp'], dayfirst=True)
                df.set_index('timestamp', inplace=True)

        # Outer join per no perdre cap esdeveniment clínic
        df_pacient = df_g.copy()
        if not df_m.empty:
            df_pacient = df_pacient.join(df_m, how='outer')
        if not df_b.empty:
            df_pacient = df_pacient.join(df_b, how='outer')

        df_pacient = df_pacient.sort_index()

        # Remostratge a 5 minuts
        agg_dict = {'glucose': 'mean'}
        if 'carbs' in df_pacient.columns:
            agg_dict['carbs'] = 'sum'
        if 'bolus' in df_pacient.columns:
            agg_dict['bolus'] = 'sum'

        df_resampled = df_pacient.resample('5min').agg(agg_dict)

        # Imputació lineal: màxim 3 mostres consecutives (15 min)
        df_resampled['glucose_imputed'] = df_resampled['glucose'].interpolate(
            method='linear', limit=3, limit_direction='forward'
        )

        # Emplenar zeros per a carbohidrats i insulina
        if 'carbs' in df_resampled.columns:
            df_resampled['carbs'] = df_resampled['carbs'].fillna(0.0)
        else:
            df_resampled['carbs'] = 0.0

        if 'bolus' in df_resampled.columns:
            df_resampled['bolus'] = df_resampled['bolus'].fillna(0.0)
        else:
            df_resampled['bolus'] = 0.0

        return df_resampled

    except Exception as e:
        print(f"  ERROR en processar '{ruta_arxiu}': {e}")
        return None

### 2. Detecció de Segments Continus

Implemente la lògica de segmentació per a garantir la continuïtat temporal exigida pel filtre de Kalman i els models autoregressius. Qualsevol pèrdua de senyal superior a 15 minuts actua com a punt de ruptura. La funció analitza aquestes ruptures i descarta els blocs resultants que no superen el llindar operatiu definit de 12 hores.

In [ ]:
def segmentar_serie(df_resampled, llindar_mostres, pacient_id):
    segments_valids = []

    # Màscara: True on la glucosa imputada és NaN (buit no rescatable)
    mask_nans = df_resampled['glucose_imputed'].isna()

    # Assignar ID de segment: cada canvi de NaN a valor o de valor a NaN crea un nou grup
    canvis = mask_nans != mask_nans.shift(fill_value=not mask_nans.iloc[0])
    df_resampled = df_resampled.copy()
    df_resampled['_segment_id'] = canvis.cumsum()

    # Descartem les files on la glucosa imputada és NaN
    df_clean = df_resampled.dropna(subset=['glucose_imputed']).copy()

    # Extraiem segments vàlids
    for seg_id, grup in df_clean.groupby('_segment_id'):
        if len(grup) >= llindar_mostres:
            seg_export = grup.drop(columns=['_segment_id']).copy()
            seg_export['pacient_id'] = pacient_id
            segments_valids.append(seg_export)

    return segments_valids

### 3. Orquestració i Traçabilitat

Cree una funció principal que s'encarrega d'iterar sobre tots els fitxers d'un determinat subconjunt (entrenament o prova). Aquesta funció coordina l'extracció i la segmentació, assigna identificadors únics a cada segment vàlid per a mantenir la traçabilitat de les dades i genera un registre de control de qualitat per a documentar les pèrdues d'informació.

In [4]:
def processar_split(glob_pattern, llindar_mostres, nom_split):
    print(f"\n{'='*60}")
    print(f"Processant conjunt de {nom_split.upper()}...")
    print(f"Llindar mínim per segment: {llindar_mostres} mostres "
          f"({llindar_mostres * 5 // 60}h {llindar_mostres * 5 % 60}min)")
    print(f"{'='*60}")

    lista_arxius = sorted(glob.glob(glob_pattern))
    if not lista_arxius:
        print(f"ADVERTÈNCIA: No s'han trobat fitxers amb el patró '{glob_pattern}'")
        return None, None

    tots_els_segments = []
    resum_qualitat    = []

    for ruta in lista_arxius:
        nom_arxiu  = os.path.basename(ruta)
        pacient_id = nom_arxiu.split('-')[0]

        print(f"\n  → Pacient {pacient_id}...")

        df_resampled = processar_arxiu_xml(ruta)
        if df_resampled is None:
            print(f"     SALTAT (error en la càrrega)")
            continue

        # Estadístiques de buits ABANS d'imputar
        df_glucosa_original = df_resampled['glucose'].dropna().sort_index()
        diff_min = df_glucosa_original.index.to_series().diff().dt.total_seconds() / 60
        buits    = diff_min[diff_min > 7]  

        # Segmentació i filtrat
        segments = segmentar_serie(df_resampled, llindar_mostres, pacient_id)

        for i, seg in enumerate(segments):
            seg['segment_num'] = i + 1
            tots_els_segments.append(seg)

        mostres_totals = sum(len(s) for s in segments)
        resum_qualitat.append({
            'Pacient':           pacient_id,
            'Registres_bruts':   len(df_glucosa_original),
            'Buits_detectats':   len(buits),
            'Max_buit_hores':    round(buits.max() / 60, 2) if len(buits) > 0 else 0,
            'Segments_valids':   len(segments),
            'Mostres_netes':     mostres_totals,
            'Hores_netes':       round(mostres_totals * 5 / 60, 1)
        })

        print(f"     Buits detectats: {len(buits)}  |  "
              f"Segments vàlids: {len(segments)}  |  "
              f"Mostres netes: {mostres_totals} ({round(mostres_totals * 5 / 60, 1)}h)")

    if not tots_els_segments:
        print(f"\nERROR: No s'han extret segments vàlids per al conjunt de {nom_split}.")
        return None, pd.DataFrame(resum_qualitat)

    df_final = pd.concat(tots_els_segments)
    df_final.reset_index(inplace=True)
    df_final.rename(columns={'index': 'timestamp'}, inplace=True)

    cols_ordre = ['timestamp', 'pacient_id', 'segment_num',
                  'glucose', 'glucose_imputed', 'carbs', 'bolus']
    cols_existents = [c for c in cols_ordre if c in df_final.columns]
    df_final = df_final[cols_existents]

    return df_final, pd.DataFrame(resum_qualitat)

### 4. Execució Global i Exportació de Datasets

S'executa el *pipeline* de processament per als conjunts d'entrenament i prova de manera independent. S'exporten els resultats a fitxers CSV finals, els quals formaran la base de dades íntegra i lliure d'errors estructurals sobre la qual s'entrenaran i avaluaran els models d'aprenentatge.

In [5]:
# Execució per a l'entrenament
df_train, resum_train = processar_split(
    glob_pattern    = RUTA_ARXIUS_TRAIN,
    llindar_mostres = LLINDAR_TRAIN,
    nom_split       = "training"
)

if df_train is not None:
    ruta_csv_train = os.path.join(RUTA_EIXIDA, "dataset_entrenament_net.csv")
    df_train.to_csv(ruta_csv_train, index=False)
    print(f"\n--- RESUM CONJUNT TRAINING ---")
    print(resum_train.to_string(index=False))
    print(f"\nTotal registres exportats: {len(df_train)}")
    print(f"Fitxer guardat: '{ruta_csv_train}'")

# Execució per al test
df_test, resum_test = processar_split(
    glob_pattern    = RUTA_ARXIUS_TEST,
    llindar_mostres = LLINDAR_TEST,
    nom_split       = "testing"
)

if df_test is not None:
    ruta_csv_test = os.path.join(RUTA_EIXIDA, "dataset_test_net.csv")
    df_test.to_csv(ruta_csv_test, index=False)
    print(f"\n--- RESUM CONJUNT TESTING ---")
    print(resum_test.to_string(index=False))
    print(f"\nTotal registres exportats: {len(df_test)}")
    print(f"Fitxer guardat: '{ruta_csv_test}'")

# Resum final per consola
print("\n" + "="*60)
print("PREPROCESSAMENT COMPLETAT I EXPORTAT")
print("="*60)
print(f"Els conjunts de dades nets s'han generat correctament a '{RUTA_EIXIDA}/'.")


Processant conjunt de TRAINING...
Llindar mínim per segment: 144 mostres (12h 0min)

  → Pacient 559...
     Buits detectats: 42  |  Segments vàlids: 31  |  Mostres netes: 10094 (841.2h)

  → Pacient 563...
     Buits detectats: 21  |  Segments vàlids: 13  |  Mostres netes: 12022 (1001.8h)

  → Pacient 570...
     Buits detectats: 20  |  Segments vàlids: 16  |  Mostres netes: 10821 (901.8h)

  → Pacient 575...
     Buits detectats: 72  |  Segments vàlids: 41  |  Mostres netes: 11150 (929.2h)

  → Pacient 588...
     Buits detectats: 10  |  Segments vàlids: 8  |  Mostres netes: 12451 (1037.6h)

  → Pacient 591...
     Buits detectats: 26  |  Segments vàlids: 17  |  Mostres netes: 10501 (875.1h)

--- RESUM CONJUNT TRAINING ---
Pacient  Registres_bruts  Buits_detectats  Max_buit_hores  Segments_valids  Mostres_netes  Hores_netes
    559            10796               42           13.08               31          10094        841.2
    563            12124               21           24.08 